In [2]:
import os
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

path = kagglehub.dataset_download("miadul/e-commerce-sales-transactions-dataset")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
file_path = os.path.join(path, csv_files[0])

df = pd.read_csv(file_path)

df['order_date'] = pd.to_datetime(df['order_date'])
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month_name()
df['day_of_week'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

df['discount_amount'] = (df['price'] * df['quantity']) * df['discount']
df['net_profit'] = df['total_amount'] * (df['profit_margin'] / 100)

max_date = df['order_date'].max()
rfm = df.groupby('customer_id').agg(
    recency=('order_date', lambda x: (max_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_amount', 'sum')
).reset_index()

rfm['R_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4])

rfm['rfm_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

def segment_customer(df_rfm):
    score = int(df_rfm['rfm_score'])
    if score >= 444:
        return 'VIP / Champions'
    elif score >= 333:
        return 'Loyal Customers'
    elif score >= 222:
        return 'At Risk'
    else:
        return 'Hibernating / Lost'

rfm['customer_segment'] = rfm.apply(segment_customer, axis=1)

df = df.merge(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'customer_segment']], on='customer_id', how='left')

df_ml = df.copy()
df_ml['returned_flag'] = (df_ml['returned'] == 'Yes').astype(int)

categorical_cols = ['category', 'payment_method', 'region', 'customer_gender']
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col])
    encoders[col] = le

features = [
    'price', 'discount', 'quantity', 'delivery_time_days', 
    'shipping_cost', 'customer_age', 'category_enc', 
    'payment_method_enc', 'region_enc', 'customer_gender_enc'
]

X = df_ml[features]
y = df_ml['returned_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

df['return_probability'] = model.predict_proba(X[features])[:, 1]
df['predicted_return'] = np.where(df['return_probability'] >= 0.5, 'High Risk', 'Low Risk')

output_filename = 'enriched_ecommerce_dataset.csv'
df.to_csv(output_filename, index=False)

print("Processing finished successfully.")
print(f"Dataset saved as: {output_filename}")
print(df[['order_id', 'customer_segment', 'return_probability', 'predicted_return']].head())

Processing finished successfully.
Dataset saved as: enriched_ecommerce_dataset.csv
  order_id    customer_segment  return_probability predicted_return
0  O100000  Hibernating / Lost                0.01         Low Risk
1  O100001     Loyal Customers                0.01         Low Risk
2  O100002     Loyal Customers                0.01         Low Risk
3  O100003     Loyal Customers                0.04         Low Risk
4  O100004             At Risk                0.00         Low Risk
